In [1]:
silver_tables = [
    "learner_profiles",
    "question_bank",
    "reference_materials",
    "learning_events",
    "practice_attempts",
    "pre_practice_feedback",
    "post_practice_feedback",
    "learner_check_in",
    "learner_check_in_topics",
    "ai_extracted_insights",
    "validated_learning_insights",
]

silver_row_counts = []

for table_name in silver_tables:
    full_table_name = f"demo.silver.{table_name}"

    row_count = spark.table(full_table_name).count()

    silver_row_counts.append(
        (table_name, row_count)
    )

silver_row_counts_df = spark.createDataFrame(
    silver_row_counts,
    ["table_name", "row_count"]
)

silver_row_counts_df.show(truncate=False)

+---------------------------+---------+
|table_name                 |row_count|
+---------------------------+---------+
|learner_profiles           |4        |
|question_bank              |5        |
|reference_materials        |6        |
|learning_events            |6        |
|practice_attempts          |5        |
|pre_practice_feedback      |3        |
|post_practice_feedback     |3        |
|learner_check_in           |3        |
|learner_check_in_topics    |5        |
|ai_extracted_insights      |9        |
|validated_learning_insights|0        |
+---------------------------+---------+



In [2]:
expected_silver_counts = {
    "learner_profiles": 4,
    "question_bank": 5,
    "reference_materials": 6,
    "learning_events": 6,
    "practice_attempts": 5,
    "pre_practice_feedback": 3,
    "post_practice_feedback": 3,
    "learner_check_in": 3,
    "learner_check_in_topics": 5,
    "ai_extracted_insights": 9,
    "validated_learning_insights": 0,
}

count_check_results = []

for table_name, actual_count in silver_row_counts:
    expected_count = expected_silver_counts[table_name]

    status = (
        "PASS"
        if actual_count == expected_count
        else "FAIL"
    )

    count_check_results.append(
        (
            table_name,
            expected_count,
            actual_count,
            status,
        )
    )

count_check_results_df = spark.createDataFrame(
    count_check_results,
    [
        "table_name",
        "expected_count",
        "actual_count",
        "status",
    ]
)

count_check_results_df.show(truncate=False)

+---------------------------+--------------+------------+------+
|table_name                 |expected_count|actual_count|status|
+---------------------------+--------------+------------+------+
|learner_profiles           |4             |4           |PASS  |
|question_bank              |5             |5           |PASS  |
|reference_materials        |6             |6           |PASS  |
|learning_events            |6             |6           |PASS  |
|practice_attempts          |5             |5           |PASS  |
|pre_practice_feedback      |3             |3           |PASS  |
|post_practice_feedback     |3             |3           |PASS  |
|learner_check_in           |3             |3           |PASS  |
|learner_check_in_topics    |5             |5           |PASS  |
|ai_extracted_insights      |9             |9           |PASS  |
|validated_learning_insights|0             |0           |PASS  |
+---------------------------+--------------+------------+------+



In [3]:
spark.sql("""
SELECT
    COUNT(*) AS total_rows,

    SUM(
        CASE
            WHEN user_id IS NULL
              OR TRIM(user_id) = ''
            THEN 1 ELSE 0
        END
    ) AS invalid_user_id,

    SUM(
        CASE
            WHEN registration_date IS NULL
            THEN 1 ELSE 0
        END
    ) AS null_registration_date,

    SUM(
        CASE
            WHEN profile_updated_at IS NULL
            THEN 1 ELSE 0
        END
    ) AS null_profile_updated_at,

    SUM(
        CASE
            WHEN ingestion_time IS NULL
            THEN 1 ELSE 0
        END
    ) AS null_ingestion_time,

    SUM(
        CASE
            WHEN registration_date > TO_DATE(profile_updated_at)
            THEN 1 ELSE 0
        END
    ) AS registration_after_profile_update,

    SUM(
        CASE
            WHEN ingestion_time < profile_updated_at
            THEN 1 ELSE 0
        END
    ) AS invalid_time_order

FROM demo.silver.learner_profiles
""").show()

+----------+---------------+----------------------+-----------------------+-------------------+---------------------------------+------------------+
|total_rows|invalid_user_id|null_registration_date|null_profile_updated_at|null_ingestion_time|registration_after_profile_update|invalid_time_order|
+----------+---------------+----------------------+-----------------------+-------------------+---------------------------------+------------------+
|         4|              0|                     0|                      0|                  0|                                0|                 0|
+----------+---------------+----------------------+-----------------------+-------------------+---------------------------------+------------------+



In [4]:
spark.sql("""
SELECT
    user_id,
    profile_updated_at,
    COUNT(*) AS duplicate_count
FROM demo.silver.learner_profiles
GROUP BY
    user_id,
    profile_updated_at
HAVING COUNT(*) > 1
""").show()

+-------+------------------+---------------+
|user_id|profile_updated_at|duplicate_count|
+-------+------------------+---------------+
+-------+------------------+---------------+



In [5]:
spark.sql("""
SELECT
    user_id,
    SUM(CASE WHEN is_current = true THEN 1 ELSE 0 END)
        AS current_profile_count
FROM demo.silver.learner_profiles
GROUP BY user_id
HAVING current_profile_count <> 1
""").show()

+-------+---------------------+
|user_id|current_profile_count|
+-------+---------------------+
+-------+---------------------+



In [6]:
spark.sql("""
SELECT
    COUNT(*) AS total_rows,

    SUM(
        CASE
            WHEN question_id IS NULL
              OR TRIM(question_id) = ''
            THEN 1 ELSE 0
        END
    ) AS invalid_question_id,

    SUM(
        CASE
            WHEN question_version IS NULL
              OR question_version <= 0
            THEN 1 ELSE 0
        END
    ) AS invalid_question_version,

    SUM(
        CASE
            WHEN question_text IS NULL
              OR TRIM(question_text) = ''
            THEN 1 ELSE 0
        END
    ) AS empty_question_text,

    SUM(
        CASE
            WHEN option_a_text IS NULL OR TRIM(option_a_text) = ''
              OR option_b_text IS NULL OR TRIM(option_b_text) = ''
              OR option_c_text IS NULL OR TRIM(option_c_text) = ''
              OR option_d_text IS NULL OR TRIM(option_d_text) = ''
            THEN 1 ELSE 0
        END
    ) AS missing_options,

    SUM(
        CASE
            WHEN correct_option_letter NOT IN ('A', 'B', 'C', 'D')
              OR correct_option_letter IS NULL
            THEN 1 ELSE 0
        END
    ) AS invalid_correct_option,

    SUM(
        CASE
            WHEN difficulty_level IS NULL
              OR difficulty_level NOT BETWEEN 1 AND 5
            THEN 1 ELSE 0
        END
    ) AS invalid_difficulty,

    SUM(
        CASE
            WHEN validation_status NOT IN (
                'pending',
                'approved',
                'rejected',
                'flagged'
            )
              OR validation_status IS NULL
            THEN 1 ELSE 0
        END
    ) AS invalid_validation_status,

    SUM(
        CASE
            WHEN content_hash IS NULL
              OR TRIM(content_hash) = ''
            THEN 1 ELSE 0
        END
    ) AS empty_content_hash

FROM demo.silver.question_bank
""").show()

+----------+-------------------+------------------------+-------------------+---------------+----------------------+------------------+-------------------------+------------------+
|total_rows|invalid_question_id|invalid_question_version|empty_question_text|missing_options|invalid_correct_option|invalid_difficulty|invalid_validation_status|empty_content_hash|
+----------+-------------------+------------------------+-------------------+---------------+----------------------+------------------+-------------------------+------------------+
|         5|                  0|                       0|                  0|              0|                     0|                 0|                        0|                 0|
+----------+-------------------+------------------------+-------------------+---------------+----------------------+------------------+-------------------------+------------------+



In [7]:
spark.sql("""
SELECT
    question_id,
    question_version,
    COUNT(*) AS duplicate_count
FROM demo.silver.question_bank
GROUP BY
    question_id,
    question_version
HAVING COUNT(*) > 1
""").show()

+-----------+----------------+---------------+
|question_id|question_version|duplicate_count|
+-----------+----------------+---------------+
+-----------+----------------+---------------+



In [8]:
spark.sql("""
SELECT
    content_hash,
    COUNT(*) AS duplicate_content_count
FROM demo.silver.question_bank
GROUP BY content_hash
HAVING COUNT(*) > 1
""").show()

+------------+-----------------------+
|content_hash|duplicate_content_count|
+------------+-----------------------+
+------------+-----------------------+



In [9]:
spark.sql("""
SELECT
    COUNT(*) AS total_rows,

    SUM(
        CASE
            WHEN reference_id IS NULL
              OR TRIM(reference_id) = ''
            THEN 1 ELSE 0
        END
    ) AS invalid_reference_id,

    SUM(
        CASE
            WHEN source_type IS NULL
              OR TRIM(source_type) = ''
            THEN 1 ELSE 0
        END
    ) AS invalid_source_type,

    SUM(
        CASE
            WHEN title IS NULL
              OR TRIM(title) = ''
            THEN 1 ELSE 0
        END
    ) AS empty_title,

    SUM(
        CASE
            WHEN content_text IS NULL
              OR TRIM(content_text) = ''
            THEN 1 ELSE 0
        END
    ) AS empty_content_text,

    SUM(
        CASE
            WHEN reliability_level NOT IN (
                'official',
                'approved',
                'external'
            )
              OR reliability_level IS NULL
            THEN 1 ELSE 0
        END
    ) AS invalid_reliability_level,

    SUM(
        CASE
            WHEN import_time IS NULL
            THEN 1 ELSE 0
        END
    ) AS null_import_time,

    SUM(
        CASE
            WHEN ingestion_time IS NULL
            THEN 1 ELSE 0
        END
    ) AS null_ingestion_time,

    SUM(
        CASE
            WHEN ingestion_time < import_time
            THEN 1 ELSE 0
        END
    ) AS invalid_time_order,

    SUM(
        CASE
            WHEN content_hash IS NULL
              OR TRIM(content_hash) = ''
            THEN 1 ELSE 0
        END
    ) AS empty_content_hash

FROM demo.silver.reference_materials
""").show()

+----------+--------------------+-------------------+-----------+------------------+-------------------------+----------------+-------------------+------------------+------------------+
|total_rows|invalid_reference_id|invalid_source_type|empty_title|empty_content_text|invalid_reliability_level|null_import_time|null_ingestion_time|invalid_time_order|empty_content_hash|
+----------+--------------------+-------------------+-----------+------------------+-------------------------+----------------+-------------------+------------------+------------------+
|         6|                   0|                  0|          0|                 0|                        0|               0|                  0|                 0|                 0|
+----------+--------------------+-------------------+-----------+------------------+-------------------------+----------------+-------------------+------------------+------------------+



In [10]:
spark.sql("""
SELECT
    reference_id,
    COUNT(*) AS duplicate_count
FROM demo.silver.reference_materials
GROUP BY reference_id
HAVING COUNT(*) > 1
""").show()

+------------+---------------+
|reference_id|duplicate_count|
+------------+---------------+
+------------+---------------+



In [11]:
spark.sql("""
SELECT
    content_hash,
    COUNT(*) AS duplicate_content_count
FROM demo.silver.reference_materials
GROUP BY content_hash
HAVING COUNT(*) > 1
""").show()

+------------+-----------------------+
|content_hash|duplicate_content_count|
+------------+-----------------------+
+------------+-----------------------+



In [12]:
spark.sql("""
SELECT
    COUNT(*) AS total_rows,

    SUM(
        CASE
            WHEN event_id IS NULL
              OR TRIM(event_id) = ''
            THEN 1 ELSE 0
        END
    ) AS invalid_event_id,

    SUM(
        CASE
            WHEN user_id IS NULL
              OR TRIM(user_id) = ''
            THEN 1 ELSE 0
        END
    ) AS invalid_user_id,

    SUM(
        CASE
            WHEN session_id IS NULL
              OR TRIM(session_id) = ''
            THEN 1 ELSE 0
        END
    ) AS invalid_session_id,

    SUM(
        CASE
            WHEN event_type NOT IN (
                'ai_learning_interaction',
                'practice_submitted'
            )
              OR event_type IS NULL
            THEN 1 ELSE 0
        END
    ) AS invalid_event_type,

    SUM(
        CASE
            WHEN event_time IS NULL
            THEN 1 ELSE 0
        END
    ) AS null_event_time,

    SUM(
        CASE
            WHEN ingestion_time IS NULL
            THEN 1 ELSE 0
        END
    ) AS null_ingestion_time,

    SUM(
        CASE
            WHEN ingestion_time < event_time
            THEN 1 ELSE 0
        END
    ) AS invalid_time_order,

    SUM(
        CASE
            WHEN event_date <> TO_DATE(event_time)
            THEN 1 ELSE 0
        END
    ) AS invalid_event_date,

    SUM(
        CASE
            WHEN event_hour <> HOUR(event_time)
            THEN 1 ELSE 0
        END
    ) AS invalid_event_hour,

    SUM(
        CASE
            WHEN payload_valid <> true
            THEN 1 ELSE 0
        END
    ) AS invalid_payload,

    SUM(
        CASE
            WHEN processing_status <> 'processed'
            THEN 1 ELSE 0
        END
    ) AS invalid_processing_status

FROM demo.silver.learning_events
""").show()

+----------+----------------+---------------+------------------+------------------+---------------+-------------------+------------------+------------------+------------------+---------------+-------------------------+
|total_rows|invalid_event_id|invalid_user_id|invalid_session_id|invalid_event_type|null_event_time|null_ingestion_time|invalid_time_order|invalid_event_date|invalid_event_hour|invalid_payload|invalid_processing_status|
+----------+----------------+---------------+------------------+------------------+---------------+-------------------+------------------+------------------+------------------+---------------+-------------------------+
|         6|               0|              0|                 0|                 0|              0|                  0|                 0|                 0|                 0|              0|                        0|
+----------+----------------+---------------+------------------+------------------+---------------+-------------------+-----

In [13]:
spark.sql("""
SELECT
    event_id,
    COUNT(*) AS duplicate_count
FROM demo.silver.learning_events
GROUP BY event_id
HAVING COUNT(*) > 1
""").show()

+--------+---------------+
|event_id|duplicate_count|
+--------+---------------+
+--------+---------------+



In [14]:
spark.sql("""
SELECT
    event_id,
    event_type,
    source_system
FROM demo.silver.learning_events
WHERE
    (
        event_type = 'ai_learning_interaction'
        AND source_system <> 'chat'
    )
    OR
    (
        event_type = 'practice_submitted'
        AND source_system <> 'practice_app'
    )
""").show()

+--------+----------+-------------+
|event_id|event_type|source_system|
+--------+----------+-------------+
+--------+----------+-------------+



In [15]:
spark.sql("""
SELECT
    COUNT(*) AS total_rows,

    SUM(
        CASE
            WHEN attempt_id IS NULL
              OR TRIM(attempt_id) = ''
            THEN 1 ELSE 0
        END
    ) AS invalid_attempt_id,

    SUM(
        CASE
            WHEN event_id IS NULL
              OR TRIM(event_id) = ''
            THEN 1 ELSE 0
        END
    ) AS invalid_event_id,

    SUM(
        CASE
            WHEN user_id IS NULL
              OR TRIM(user_id) = ''
            THEN 1 ELSE 0
        END
    ) AS invalid_user_id,

    SUM(
        CASE
            WHEN practice_id IS NULL
              OR TRIM(practice_id) = ''
            THEN 1 ELSE 0
        END
    ) AS invalid_practice_id,

    SUM(
        CASE
            WHEN question_id IS NULL
              OR TRIM(question_id) = ''
            THEN 1 ELSE 0
        END
    ) AS invalid_question_id,

    SUM(
        CASE
            WHEN question_version IS NULL
              OR question_version <= 0
            THEN 1 ELSE 0
        END
    ) AS invalid_question_version,

    SUM(
        CASE
            WHEN selected_option_letter NOT IN ('A', 'B', 'C', 'D')
              OR selected_option_letter IS NULL
            THEN 1 ELSE 0
        END
    ) AS invalid_selected_option,

    SUM(
        CASE
            WHEN score NOT IN (0.0, 1.0)
              OR score IS NULL
            THEN 1 ELSE 0
        END
    ) AS invalid_score,

    SUM(
        CASE
            WHEN hints_used IS NULL
              OR hints_used < 0
            THEN 1 ELSE 0
        END
    ) AS invalid_hints_used,

    SUM(
        CASE
            WHEN attempt_duration_seconds IS NULL
              OR attempt_duration_seconds < 0
            THEN 1 ELSE 0
        END
    ) AS invalid_attempt_duration,

    SUM(
        CASE
            WHEN attempt_number IS NULL
              OR attempt_number <= 0
            THEN 1 ELSE 0
        END
    ) AS invalid_attempt_number

FROM demo.silver.practice_attempts
""").show()

+----------+------------------+----------------+---------------+-------------------+-------------------+------------------------+-----------------------+-------------+------------------+------------------------+----------------------+
|total_rows|invalid_attempt_id|invalid_event_id|invalid_user_id|invalid_practice_id|invalid_question_id|invalid_question_version|invalid_selected_option|invalid_score|invalid_hints_used|invalid_attempt_duration|invalid_attempt_number|
+----------+------------------+----------------+---------------+-------------------+-------------------+------------------------+-----------------------+-------------+------------------+------------------------+----------------------+
|         5|                 0|               0|              0|                  0|                  0|                       0|                      0|            0|                 0|                       0|                     0|
+----------+------------------+----------------+------------

In [16]:
spark.sql("""
SELECT
    attempt_id,
    COUNT(*) AS duplicate_count
FROM demo.silver.practice_attempts
GROUP BY attempt_id
HAVING COUNT(*) > 1
""").show()

+----------+---------------+
|attempt_id|duplicate_count|
+----------+---------------+
+----------+---------------+



In [17]:
spark.sql("""
SELECT
    pa.attempt_id,
    pa.event_id
FROM demo.silver.practice_attempts pa
LEFT JOIN demo.silver.learning_events le
    ON pa.event_id = le.event_id
WHERE
    le.event_id IS NULL
    OR le.event_type <> 'practice_submitted'
""").show()

+----------+--------+
|attempt_id|event_id|
+----------+--------+
+----------+--------+



In [18]:
spark.sql("""
SELECT
    pa.attempt_id,
    pa.question_id,
    pa.question_version
FROM demo.silver.practice_attempts pa
LEFT JOIN demo.silver.question_bank qb
    ON pa.question_id = qb.question_id
   AND pa.question_version = qb.question_version
WHERE qb.question_id IS NULL
""").show()

+----------+-----------+----------------+
|attempt_id|question_id|question_version|
+----------+-----------+----------------+
+----------+-----------+----------------+



In [19]:
spark.sql("""
SELECT
    pa.attempt_id,
    pa.question_id,
    pa.selected_option_letter,
    qb.correct_option_letter,
    pa.is_correct,
    pa.score
FROM demo.silver.practice_attempts pa
JOIN demo.silver.question_bank qb
    ON pa.question_id = qb.question_id
   AND pa.question_version = qb.question_version
WHERE
    pa.is_correct <>
        (pa.selected_option_letter = qb.correct_option_letter)
    OR pa.score <>
        CASE
            WHEN pa.selected_option_letter = qb.correct_option_letter
            THEN 1.0
            ELSE 0.0
        END
""").show()

+----------+-----------+----------------------+---------------------+----------+-----+
|attempt_id|question_id|selected_option_letter|correct_option_letter|is_correct|score|
+----------+-----------+----------------------+---------------------+----------+-----+
+----------+-----------+----------------------+---------------------+----------+-----+



In [20]:
spark.sql("""
SELECT
    COUNT(*) AS total_rows,

    SUM(CASE
        WHEN feedback_id IS NULL OR TRIM(feedback_id) = ''
        THEN 1 ELSE 0
    END) AS invalid_feedback_id,

    SUM(CASE
        WHEN user_id IS NULL OR TRIM(user_id) = ''
        THEN 1 ELSE 0
    END) AS invalid_user_id,

    SUM(CASE
        WHEN practice_id IS NULL OR TRIM(practice_id) = ''
        THEN 1 ELSE 0
    END) AS invalid_practice_id,

    SUM(CASE
        WHEN feedback_time IS NULL
        THEN 1 ELSE 0
    END) AS null_feedback_time,

    SUM(CASE
        WHEN ingestion_time IS NULL
        THEN 1 ELSE 0
    END) AS null_ingestion_time,

    SUM(CASE
        WHEN ingestion_time < feedback_time
        THEN 1 ELSE 0
    END) AS invalid_time_order,

    SUM(CASE
        WHEN delay_minutes IS NULL OR delay_minutes < 0
        THEN 1 ELSE 0
    END) AS invalid_delay_minutes,

    SUM(CASE
        WHEN confidence_before_score NOT BETWEEN 1 AND 10
          OR confidence_before_score IS NULL
        THEN 1 ELSE 0
    END) AS invalid_confidence_before,

    SUM(CASE
        WHEN perceived_understanding_before_score NOT BETWEEN 1 AND 10
          OR perceived_understanding_before_score IS NULL
        THEN 1 ELSE 0
    END) AS invalid_understanding_before,

    SUM(CASE
        WHEN expected_difficulty_score NOT BETWEEN 1 AND 10
          OR expected_difficulty_score IS NULL
        THEN 1 ELSE 0
    END) AS invalid_expected_difficulty

FROM demo.silver.pre_practice_feedback
""").show()

+----------+-------------------+---------------+-------------------+------------------+-------------------+------------------+---------------------+-------------------------+----------------------------+---------------------------+
|total_rows|invalid_feedback_id|invalid_user_id|invalid_practice_id|null_feedback_time|null_ingestion_time|invalid_time_order|invalid_delay_minutes|invalid_confidence_before|invalid_understanding_before|invalid_expected_difficulty|
+----------+-------------------+---------------+-------------------+------------------+-------------------+------------------+---------------------+-------------------------+----------------------------+---------------------------+
|         3|                  0|              0|                  0|                 0|                  0|                 0|                    0|                        0|                           0|                          0|
+----------+-------------------+---------------+-------------------+----

In [21]:
spark.sql("""
SELECT
    COUNT(*) AS total_rows,

    SUM(CASE
        WHEN feedback_id IS NULL OR TRIM(feedback_id) = ''
        THEN 1 ELSE 0
    END) AS invalid_feedback_id,

    SUM(CASE
        WHEN user_id IS NULL OR TRIM(user_id) = ''
        THEN 1 ELSE 0
    END) AS invalid_user_id,

    SUM(CASE
        WHEN practice_id IS NULL OR TRIM(practice_id) = ''
        THEN 1 ELSE 0
    END) AS invalid_practice_id,

    SUM(CASE
        WHEN ingestion_time < feedback_time
        THEN 1 ELSE 0
    END) AS invalid_time_order,

    SUM(CASE
        WHEN delay_minutes IS NULL OR delay_minutes < 0
        THEN 1 ELSE 0
    END) AS invalid_delay_minutes,

    SUM(CASE
        WHEN confidence_after_score NOT BETWEEN 1 AND 10
          OR confidence_after_score IS NULL
        THEN 1 ELSE 0
    END) AS invalid_confidence_after,

    SUM(CASE
        WHEN perceived_understanding_after_score NOT BETWEEN 1 AND 10
          OR perceived_understanding_after_score IS NULL
        THEN 1 ELSE 0
    END) AS invalid_understanding_after,

    SUM(CASE
        WHEN perceived_difficulty_score NOT BETWEEN 1 AND 10
          OR perceived_difficulty_score IS NULL
        THEN 1 ELSE 0
    END) AS invalid_perceived_difficulty,

    SUM(CASE
        WHEN still_confused IS NULL
        THEN 1 ELSE 0
    END) AS null_still_confused

FROM demo.silver.post_practice_feedback
""").show()

+----------+-------------------+---------------+-------------------+------------------+---------------------+------------------------+---------------------------+----------------------------+-------------------+
|total_rows|invalid_feedback_id|invalid_user_id|invalid_practice_id|invalid_time_order|invalid_delay_minutes|invalid_confidence_after|invalid_understanding_after|invalid_perceived_difficulty|null_still_confused|
+----------+-------------------+---------------+-------------------+------------------+---------------------+------------------------+---------------------------+----------------------------+-------------------+
|         3|                  0|              0|                  0|                 0|                    0|                       0|                          0|                           0|                  0|
+----------+-------------------+---------------+-------------------+------------------+---------------------+------------------------+------------------

In [22]:
spark.sql("""
SELECT feedback_id, COUNT(*) AS duplicate_count
FROM demo.silver.pre_practice_feedback
GROUP BY feedback_id
HAVING COUNT(*) > 1
""").show()

spark.sql("""
SELECT feedback_id, COUNT(*) AS duplicate_count
FROM demo.silver.post_practice_feedback
GROUP BY feedback_id
HAVING COUNT(*) > 1
""").show()

+-----------+---------------+
|feedback_id|duplicate_count|
+-----------+---------------+
+-----------+---------------+

+-----------+---------------+
|feedback_id|duplicate_count|
+-----------+---------------+
+-----------+---------------+



In [23]:
spark.sql("""
SELECT
    COUNT(*) AS total_rows,

    SUM(CASE
        WHEN feedback_id IS NULL OR TRIM(feedback_id) = ''
        THEN 1 ELSE 0
    END) AS invalid_feedback_id,

    SUM(CASE
        WHEN user_id IS NULL OR TRIM(user_id) = ''
        THEN 1 ELSE 0
    END) AS invalid_user_id,

    SUM(CASE
        WHEN feedback_time IS NULL
        THEN 1 ELSE 0
    END) AS null_feedback_time,

    SUM(CASE
        WHEN ingestion_time IS NULL
        THEN 1 ELSE 0
    END) AS null_ingestion_time,

    SUM(CASE
        WHEN ingestion_time < feedback_time
        THEN 1 ELSE 0
    END) AS invalid_time_order,

    SUM(CASE
        WHEN delay_minutes IS NULL OR delay_minutes < 0
        THEN 1 ELSE 0
    END) AS invalid_delay_minutes,

    SUM(CASE
        WHEN overall_confidence_score NOT BETWEEN 1 AND 10
          OR overall_confidence_score IS NULL
        THEN 1 ELSE 0
    END) AS invalid_overall_confidence,

    SUM(CASE
        WHEN overall_motivation_score NOT BETWEEN 1 AND 10
          OR overall_motivation_score IS NULL
        THEN 1 ELSE 0
    END) AS invalid_overall_motivation,

    SUM(CASE
        WHEN overall_stress_score NOT BETWEEN 1 AND 10
          OR overall_stress_score IS NULL
        THEN 1 ELSE 0
    END) AS invalid_overall_stress

FROM demo.silver.learner_check_in
""").show()

+----------+-------------------+---------------+------------------+-------------------+------------------+---------------------+--------------------------+--------------------------+----------------------+
|total_rows|invalid_feedback_id|invalid_user_id|null_feedback_time|null_ingestion_time|invalid_time_order|invalid_delay_minutes|invalid_overall_confidence|invalid_overall_motivation|invalid_overall_stress|
+----------+-------------------+---------------+------------------+-------------------+------------------+---------------------+--------------------------+--------------------------+----------------------+
|         3|                  0|              0|                 0|                  0|                 0|                    0|                         0|                         0|                     0|
+----------+-------------------+---------------+------------------+-------------------+------------------+---------------------+--------------------------+---------------------

In [24]:
spark.sql("""
SELECT
    COUNT(*) AS total_rows,

    SUM(CASE
        WHEN feedback_id IS NULL OR TRIM(feedback_id) = ''
        THEN 1 ELSE 0
    END) AS invalid_feedback_id,

    SUM(CASE
        WHEN user_id IS NULL OR TRIM(user_id) = ''
        THEN 1 ELSE 0
    END) AS invalid_user_id,

    SUM(CASE
        WHEN topic_id IS NULL OR TRIM(topic_id) = ''
        THEN 1 ELSE 0
    END) AS invalid_topic_id,

    SUM(CASE
        WHEN feedback_time IS NULL
        THEN 1 ELSE 0
    END) AS null_feedback_time,

    SUM(CASE
        WHEN perceived_understanding_score NOT BETWEEN 1 AND 10
          OR perceived_understanding_score IS NULL
        THEN 1 ELSE 0
    END) AS invalid_understanding_score,

    SUM(CASE
        WHEN topic_confidence_score NOT BETWEEN 1 AND 10
          OR topic_confidence_score IS NULL
        THEN 1 ELSE 0
    END) AS invalid_topic_confidence,

    SUM(CASE
        WHEN still_confused IS NULL
        THEN 1 ELSE 0
    END) AS null_still_confused

FROM demo.silver.learner_check_in_topics
""").show()

+----------+-------------------+---------------+----------------+------------------+---------------------------+------------------------+-------------------+
|total_rows|invalid_feedback_id|invalid_user_id|invalid_topic_id|null_feedback_time|invalid_understanding_score|invalid_topic_confidence|null_still_confused|
+----------+-------------------+---------------+----------------+------------------+---------------------------+------------------------+-------------------+
|         5|                  0|              0|               0|                 0|                          0|                       0|                  0|
+----------+-------------------+---------------+----------------+------------------+---------------------------+------------------------+-------------------+



In [25]:
spark.sql("""
SELECT
    feedback_id,
    COUNT(*) AS duplicate_count
FROM demo.silver.learner_check_in
GROUP BY feedback_id
HAVING COUNT(*) > 1
""").show()

+-----------+---------------+
|feedback_id|duplicate_count|
+-----------+---------------+
+-----------+---------------+



In [26]:
spark.sql("""
SELECT
    feedback_id,
    topic_id,
    COUNT(*) AS duplicate_count
FROM demo.silver.learner_check_in_topics
GROUP BY
    feedback_id,
    topic_id
HAVING COUNT(*) > 1
""").show()

+-----------+--------+---------------+
|feedback_id|topic_id|duplicate_count|
+-----------+--------+---------------+
+-----------+--------+---------------+



In [27]:
spark.sql("""
SELECT
    topics.feedback_id,
    topics.user_id,
    topics.session_id,
    topics.topic_id
FROM demo.silver.learner_check_in_topics topics
LEFT JOIN demo.silver.learner_check_in parent
    ON topics.feedback_id = parent.feedback_id
WHERE
    parent.feedback_id IS NULL
    OR topics.user_id <> parent.user_id
    OR NOT (
        topics.session_id = parent.session_id
        OR (
            topics.session_id IS NULL
            AND parent.session_id IS NULL
        )
    )
""").show()

+-----------+-------+----------+--------+
|feedback_id|user_id|session_id|topic_id|
+-----------+-------+----------+--------+
+-----------+-------+----------+--------+



In [28]:
spark.sql("""
SELECT
    COUNT(*) AS total_rows,

    SUM(CASE
        WHEN insight_id IS NULL OR TRIM(insight_id) = ''
        THEN 1 ELSE 0
    END) AS invalid_insight_id,

    SUM(CASE
        WHEN event_id IS NULL OR TRIM(event_id) = ''
        THEN 1 ELSE 0
    END) AS invalid_event_id,

    SUM(CASE
        WHEN user_id IS NULL OR TRIM(user_id) = ''
        THEN 1 ELSE 0
    END) AS invalid_user_id,

    SUM(CASE
        WHEN dynamic_concept_name IS NULL
          OR TRIM(dynamic_concept_name) = ''
        THEN 1 ELSE 0
    END) AS invalid_dynamic_concept,

    SUM(CASE
        WHEN extracted_at IS NULL
        THEN 1 ELSE 0
    END) AS null_extracted_at,

    SUM(CASE
        WHEN extraction_confidence IS NULL
          OR extraction_confidence NOT BETWEEN 0.0 AND 1.0
        THEN 1 ELSE 0
    END) AS invalid_extraction_confidence,

    SUM(CASE
        WHEN processing_model IS NULL
          OR TRIM(processing_model) = ''
        THEN 1 ELSE 0
    END) AS invalid_processing_model,

    SUM(CASE
        WHEN validation_status NOT IN (
            'pending',
            'validated',
            'flagged',
            'rejected'
        )
          OR validation_status IS NULL
        THEN 1 ELSE 0
    END) AS invalid_validation_status,

    SUM(CASE
        WHEN ai_attributes IS NULL
          OR TRIM(ai_attributes) = ''
        THEN 1 ELSE 0
    END) AS empty_ai_attributes

FROM demo.silver.ai_extracted_insights
""").show()

+----------+------------------+----------------+---------------+-----------------------+-----------------+-----------------------------+------------------------+-------------------------+-------------------+
|total_rows|invalid_insight_id|invalid_event_id|invalid_user_id|invalid_dynamic_concept|null_extracted_at|invalid_extraction_confidence|invalid_processing_model|invalid_validation_status|empty_ai_attributes|
+----------+------------------+----------------+---------------+-----------------------+-----------------+-----------------------------+------------------------+-------------------------+-------------------+
|         9|                 0|               0|              0|                      0|                0|                            0|                       0|                        0|                  0|
+----------+------------------+----------------+---------------+-----------------------+-----------------+-----------------------------+------------------------+-------

In [29]:
spark.sql("""
SELECT
    insight_id,
    COUNT(*) AS duplicate_count
FROM demo.silver.ai_extracted_insights
GROUP BY insight_id
HAVING COUNT(*) > 1
""").show()

+----------+---------------+
|insight_id|duplicate_count|
+----------+---------------+
+----------+---------------+



In [30]:
spark.sql("""
SELECT
    ai.insight_id,
    ai.event_id
FROM demo.silver.ai_extracted_insights ai
LEFT JOIN demo.silver.learning_events le
    ON ai.event_id = le.event_id
WHERE
    le.event_id IS NULL
    OR le.event_type <> 'ai_learning_interaction'
""").show()

+----------+--------+
|insight_id|event_id|
+----------+--------+
+----------+--------+



In [31]:
spark.sql("""
SELECT
    event_id,
    dynamic_concept_name,
    COUNT(*) AS duplicate_count
FROM demo.silver.ai_extracted_insights
GROUP BY
    event_id,
    dynamic_concept_name
HAVING COUNT(*) > 1
""").show()

+--------+--------------------+---------------+
|event_id|dynamic_concept_name|duplicate_count|
+--------+--------------------+---------------+
+--------+--------------------+---------------+



In [32]:
spark.sql("""
SELECT
    COUNT(*) AS row_count
FROM demo.silver.validated_learning_insights
""").show()

+---------+
|row_count|
+---------+
|        0|
+---------+



In [1]:
spark.sql("""
CREATE TABLE IF NOT EXISTS demo.quality.silver_quality_results (
    check_id STRING,
    check_time TIMESTAMP,
    source_table STRING,
    rule_name STRING,
    severity STRING,
    total_rows BIGINT,
    failed_rows BIGINT,
    status STRING,
    action_taken STRING,
    details STRING
)
USING iceberg
""")

DataFrame[]

In [2]:
spark.sql("""
CREATE TABLE IF NOT EXISTS demo.quality.silver_quarantine (
    quarantine_id STRING,
    detected_at TIMESTAMP,
    source_table STRING,
    record_id STRING,
    failed_rule STRING,
    severity STRING,
    failure_reason STRING,
    raw_record STRING,
    raw_payload STRING,
    quarantine_status STRING
)
USING iceberg
""")

DataFrame[]

In [3]:
spark.sql("""
SHOW TABLES IN demo.quality
""").show(truncate=False)

+---------+----------------------+-----------+
|namespace|tableName             |isTemporary|
+---------+----------------------+-----------+
|quality  |bronze_quality_results|false      |
|quality  |bronze_quarantine     |false      |
|quality  |silver_quality_results|false      |
|quality  |silver_quarantine     |false      |
+---------+----------------------+-----------+

